# Домашнее задание: градиентный бустинг

**Задача** — дополнить приведенную на лекции реализацию градиентного бустинга новым функционалом.

Необходимые функции:

- В библиотечных реализациях есть параметр `subsample` (от 0 до 1), позволяющий обучать новое дерево не на всех объектах X, а на случайной подвыборке размера subsample. Добавьте его в свой класс. Введенная таким образом случайность снижает дисперсию ансамбля и предотвращает переобучение, делая деревья более разнообразными.

- Согласно той же логике можно использовать и что-то похожее на метод случайных подпространств (вспомните лекцию про случайный лес) для обучения деревьев. Добавьте параметр `colsample_bytree` (тоже от 0 до 1). При обучении каждого базового дерева нужно давать ему доступ не ко всем признакам матрицы X, а только к случайной доле.

- Добавьте свойство `feature_importances_`, которое возвращает массив с оценкой важности каждого признака. **Hint:** так как внутри используются `DecisionTreeRegressor` из sklearn, у каждого дерева уже есть атрибут `feature_importances_`. Нужно просто усреднить эти векторы по всем деревьям в ансамбле `self.trees` (с учётом learning rate).

- Добавьте нативную поддержку категориальных признаков.

**Для желающих:** реализуйте класс `GBMClassifier`, чтобы ваш бустинг мог решать не только задачу регрессии. Если сделаете рабочий классификатор, предыдущие пункты можно не выполнять.

**Проверка:** сравните результаты обучения вашего бустинга с результатами библиотечного алгоритма на каком-либо датасете. Результаты не обязаны в точности совпадать; допустимым является отклонение вниз на 0.05 по метрике R².

---

Ниже — расширение лекционного класса `MyBoost` (критерий `friedman_mse`, стартовое предсказание — среднее по `y`). Код сосредоточен в этом ноутбуке; после выполнения ячеек с реализацией запускаются эксперименты.

### Реализация: `MyBoost` и `GBMClassifier`

Запустите ячейку ниже один раз перед экспериментами.

In [1]:
import numpy as np
import pandas as pd
from sklearn.tree import DecisionTreeRegressor


def _sigmoid(z):
    return 1.0 / (1.0 + np.exp(-np.clip(z, -500, 500)))


class MyBoost:
    """Градиентный бустинг (регрессия): антиградиент y - F(x), деревья на friedman_mse."""

    def __init__(self, n=400, lr=0.05, depth=7, seed=42, subsample=1.0, colsample_bytree=1.0):
        if not 0 < subsample <= 1.0:
            raise ValueError("subsample должен быть в (0, 1].")
        if not 0 < colsample_bytree <= 1.0:
            raise ValueError("colsample_bytree должен быть в (0, 1].")
        self.n = n
        self.lr = lr
        self.depth = depth
        self.seed = seed
        self.subsample = subsample
        self.colsample_bytree = colsample_bytree
        self.trees = []
        self._feature_indices = []
        self.initial_leaf = None
        self._n_features_in_ = None
        self._feature_importances_ = None
        self._cat_encoders = None
        self._feature_names = None

    @property
    def feature_importances_(self):
        if self._feature_importances_ is None:
            raise AttributeError("Модель ещё не обучена — вызовите fit.")
        return self._feature_importances_

    def _encode_dataframe(self, X, fit):
        if fit:
            self._cat_encoders = {}
            self._feature_names = list(X.columns)
        out = np.empty((len(X), X.shape[1]), dtype=np.float64)
        for j, col in enumerate(X.columns):
            s = X[col]
            if isinstance(s.dtype, pd.CategoricalDtype) or s.dtype == object:
                if fit:
                    cats = pd.unique(s.astype(str))
                    self._cat_encoders[col] = {c: i for i, c in enumerate(cats)}
                mapping = self._cat_encoders[col]
                out[:, j] = s.astype(str).map(lambda v: mapping.get(v, -1)).astype(np.float64)
            else:
                out[:, j] = s.astype(np.float64).to_numpy()
        return out

    def _prepare_X(self, X, fit):
        if isinstance(X, pd.DataFrame):
            arr = self._encode_dataframe(X, fit=fit)
        else:
            arr = np.asarray(X, dtype=np.float64)
        if fit:
            self._n_features_in_ = arr.shape[1]
        return arr

    def _predict_tree(self, tree, feat_idx, X_full):
        return tree.predict(X_full[:, feat_idx])

    def fit(self, X, y):
        X_arr = self._prepare_X(X, fit=True)
        n_features = self._n_features_in_
        y = np.asarray(y, dtype=np.float64).ravel()
        n_samples = X_arr.shape[0]
        if len(y) != n_samples:
            raise ValueError("Размеры X и y не совпадают.")
        rng = np.random.RandomState(self.seed)
        self.initial_leaf = float(np.mean(y))
        predictions = np.full(n_samples, self.initial_leaf, dtype=np.float64)
        self.trees = []
        self._feature_indices = []
        n_rows_sub = n_samples if self.subsample >= 1.0 else max(1, int(self.subsample * n_samples))
        n_cols_sub = n_features if self.colsample_bytree >= 1.0 else max(1, int(self.colsample_bytree * n_features))
        for t in range(self.n):
            antigrad = y - predictions
            if self.subsample >= 1.0:
                row_idx = np.arange(n_samples)
            else:
                row_idx = rng.choice(n_samples, size=n_rows_sub, replace=False)
            if self.colsample_bytree >= 1.0:
                feat_idx = np.arange(n_features)
            else:
                feat_idx = rng.choice(n_features, size=n_cols_sub, replace=False)
            X_sub = X_arr[np.ix_(row_idx, feat_idx)]
            y_sub = antigrad[row_idx]
            tree = DecisionTreeRegressor(
                max_depth=self.depth,
                random_state=self.seed + t,
                criterion="friedman_mse",
            )
            tree.fit(X_sub, y_sub)
            self.trees.append(tree)
            self._feature_indices.append(feat_idx)
            predictions += self.lr * self._predict_tree(tree, feat_idx, X_arr)
        self._compute_feature_importances()
        return self

    def _compute_feature_importances(self):
        n_features = self._n_features_in_
        acc = np.zeros(n_features, dtype=np.float64)
        for tree, feat_idx in zip(self.trees, self._feature_indices):
            full = np.zeros(n_features, dtype=np.float64)
            full[feat_idx] = tree.feature_importances_
            acc += self.lr * full
        s = acc.sum()
        if s > 0:
            acc /= s
        self._feature_importances_ = acc

    def predict(self, X):
        if self.initial_leaf is None:
            raise RuntimeError("Сначала вызовите fit.")
        X_arr = self._prepare_X(X, fit=False)
        n_samples = X_arr.shape[0]
        predictions = np.full(n_samples, self.initial_leaf, dtype=np.float64)
        for tree, feat_idx in zip(self.trees, self._feature_indices):
            predictions += self.lr * self._predict_tree(tree, feat_idx, X_arr)
        return predictions


class GBMClassifier:
    """Бинарный градиентный бустинг (логлосс), метки y в {0, 1}."""

    def __init__(self, n=400, lr=0.05, depth=7, seed=42, subsample=1.0, colsample_bytree=1.0):
        if not 0 < subsample <= 1.0:
            raise ValueError("subsample должен быть в (0, 1].")
        if not 0 < colsample_bytree <= 1.0:
            raise ValueError("colsample_bytree должен быть в (0, 1].")
        self.n = n
        self.lr = lr
        self.depth = depth
        self.seed = seed
        self.subsample = subsample
        self.colsample_bytree = colsample_bytree
        self.trees = []
        self._feature_indices = []
        self.initial_log_odds = None
        self._n_features_in_ = None
        self._feature_importances_ = None
        self._cat_encoders = None
        self._feature_names = None

    @property
    def feature_importances_(self):
        if self._feature_importances_ is None:
            raise AttributeError("Модель ещё не обучена — вызовите fit.")
        return self._feature_importances_

    def _encode_dataframe(self, X, fit):
        if fit:
            self._cat_encoders = {}
            self._feature_names = list(X.columns)
        out = np.empty((len(X), X.shape[1]), dtype=np.float64)
        for j, col in enumerate(X.columns):
            s = X[col]
            if isinstance(s.dtype, pd.CategoricalDtype) or s.dtype == object:
                if fit:
                    cats = pd.unique(s.astype(str))
                    self._cat_encoders[col] = {c: i for i, c in enumerate(cats)}
                mapping = self._cat_encoders[col]
                out[:, j] = s.astype(str).map(lambda v: mapping.get(v, -1)).astype(np.float64)
            else:
                out[:, j] = s.astype(np.float64).to_numpy()
        return out

    def _prepare_X(self, X, fit):
        if isinstance(X, pd.DataFrame):
            arr = self._encode_dataframe(X, fit=fit)
        else:
            arr = np.asarray(X, dtype=np.float64)
        if fit:
            self._n_features_in_ = arr.shape[1]
        return arr

    def _predict_tree(self, tree, feat_idx, X_full):
        return tree.predict(X_full[:, feat_idx])

    def fit(self, X, y):
        X_arr = self._prepare_X(X, fit=True)
        n_features = self._n_features_in_
        y = np.asarray(y, dtype=np.float64).ravel()
        if not np.all(np.isin(y, [0, 1])):
            raise ValueError("GBMClassifier ожидает бинарные метки 0 и 1.")
        n_samples = X_arr.shape[0]
        if len(y) != n_samples:
            raise ValueError("Размеры X и y не совпадают.")
        p = float(np.clip(np.mean(y), 1e-15, 1 - 1e-15))
        self.initial_log_odds = float(np.log(p / (1 - p)))
        F = np.full(n_samples, self.initial_log_odds, dtype=np.float64)
        rng = np.random.RandomState(self.seed)
        self.trees = []
        self._feature_indices = []
        n_rows_sub = n_samples if self.subsample >= 1.0 else max(1, int(self.subsample * n_samples))
        n_cols_sub = n_features if self.colsample_bytree >= 1.0 else max(1, int(self.colsample_bytree * n_features))
        for t in range(self.n):
            prob = _sigmoid(F)
            antigrad = y - prob
            if self.subsample >= 1.0:
                row_idx = np.arange(n_samples)
            else:
                row_idx = rng.choice(n_samples, size=n_rows_sub, replace=False)
            if self.colsample_bytree >= 1.0:
                feat_idx = np.arange(n_features)
            else:
                feat_idx = rng.choice(n_features, size=n_cols_sub, replace=False)
            X_sub = X_arr[np.ix_(row_idx, feat_idx)]
            y_sub = antigrad[row_idx]
            tree = DecisionTreeRegressor(
                max_depth=self.depth,
                random_state=self.seed + t,
                criterion="friedman_mse",
            )
            tree.fit(X_sub, y_sub)
            self.trees.append(tree)
            self._feature_indices.append(feat_idx)
            F += self.lr * self._predict_tree(tree, feat_idx, X_arr)
        self._compute_feature_importances()
        return self

    def _compute_feature_importances(self):
        n_features = self._n_features_in_
        acc = np.zeros(n_features, dtype=np.float64)
        for tree, feat_idx in zip(self.trees, self._feature_indices):
            full = np.zeros(n_features, dtype=np.float64)
            full[feat_idx] = tree.feature_importances_
            acc += self.lr * full
        s = acc.sum()
        if s > 0:
            acc /= s
        self._feature_importances_ = acc

    def decision_function(self, X):
        if self.initial_log_odds is None:
            raise RuntimeError("Сначала вызовите fit.")
        X_arr = self._prepare_X(X, fit=False)
        n_samples = X_arr.shape[0]
        F = np.full(n_samples, self.initial_log_odds, dtype=np.float64)
        for tree, feat_idx in zip(self.trees, self._feature_indices):
            F += self.lr * self._predict_tree(tree, feat_idx, X_arr)
        return F

    def predict_proba(self, X):
        p = _sigmoid(self.decision_function(X))
        return np.column_stack([1 - p, p])

    def predict(self, X):
        return (self.decision_function(X) >= 0).astype(np.int64)


### Сравнение с библиотекой и демонстрация возможностей

На California Housing сравниваем с `sklearn.ensemble.GradientBoostingRegressor` (сопоставимые гиперпараметры). Затем — важности признаков, категориальный столбец в `DataFrame`, опционально — классификатор на Breast Cancer.

In [2]:
import numpy as np
import pandas as pd

from sklearn.datasets import fetch_california_housing, load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score, mean_squared_error, accuracy_score, roc_auc_score
from sklearn.ensemble import GradientBoostingRegressor, GradientBoostingClassifier


data = fetch_california_housing()
X = pd.DataFrame(data.data, columns=data.feature_names)
y = data.target

X_temp, X_test, y_temp, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
X_train, X_val, y_train, y_val = train_test_split(X_temp, y_temp, test_size=0.25, random_state=42)

N_EST = 400
LR = 0.05
DEPTH = 7
SEED = 42

my = MyBoost(n=N_EST, lr=LR, depth=DEPTH, seed=SEED, subsample=0.8, colsample_bytree=0.9)
my.fit(X_train, y_train)

sk = GradientBoostingRegressor(
    n_estimators=N_EST,
    learning_rate=LR,
    max_depth=DEPTH,
    random_state=SEED,
    subsample=0.8,
    max_features=0.9,
)
sk.fit(X_train, y_train)

for name, model in [("MyBoost", my), ("sklearn GBR", sk)]:
    pred = model.predict(X_test)
    r2 = r2_score(y_test, pred)
    rmse = float(np.sqrt(mean_squared_error(y_test, pred)))
    print(f"{name}: R² = {r2:.4f}, RMSE = {rmse:.4f}")

r2_my = r2_score(y_test, my.predict(X_test))
r2_sk = r2_score(y_test, sk.predict(X_test))
print(f"\nРазница R² (sklearn − MyBoost): {r2_sk - r2_my:.4f}  (допустимо до ~0.05 в пользу sklearn)")


print("\nТоп-10 важностей (MyBoost):")
print(pd.Series(my.feature_importances_, index=X.columns).sort_values(ascending=False).head(10))


X_cat = X_train.copy()
rng = np.random.RandomState(0)
X_cat["region_bucket"] = pd.Categorical(rng.choice(["A", "B", "C"], size=len(X_cat)))
myc = MyBoost(n=100, lr=0.05, depth=5, seed=42, subsample=1.0, colsample_bytree=1.0)
myc.fit(X_cat, y_train)
print("R² на train с категориальным столбцом:", r2_score(y_train, myc.predict(X_cat)))


bc = load_breast_cancer()
Xb = pd.DataFrame(bc.data, columns=bc.feature_names)
yb = bc.target
Xb_tr, Xb_te, yb_tr, yb_te = train_test_split(Xb, yb, test_size=0.25, random_state=42)

g_my = GBMClassifier(n=200, lr=0.05, depth=4, seed=42, subsample=0.9, colsample_bytree=0.9)
g_my.fit(Xb_tr, yb_tr)
g_sk = GradientBoostingClassifier(
    n_estimators=200,
    learning_rate=0.05,
    max_depth=4,
    random_state=42,
    subsample=0.9,
    max_features=0.9,
)
g_sk.fit(Xb_tr, yb_tr)
proba_my = g_my.predict_proba(Xb_te)[:, 1]
proba_sk = g_sk.predict_proba(Xb_te)[:, 1]
print("\nBreast Cancer — AUC (My GBMClassifier):", roc_auc_score(yb_te, proba_my))
print("Breast Cancer — AUC (sklearn GBC):      ", roc_auc_score(yb_te, proba_sk))
print("Accuracy My / sklearn:", accuracy_score(yb_te, g_my.predict(Xb_te)), "/", accuracy_score(yb_te, g_sk.predict(Xb_te)))


MyBoost: R² = 0.8393, RMSE = 0.4589
sklearn GBR: R² = 0.8400, RMSE = 0.4579

Разница R² (sklearn − MyBoost): 0.0007  (допустимо до ~0.05 в пользу sklearn)

Топ-10 важностей (MyBoost):
MedInc        0.175094
Longitude     0.151674
Latitude      0.145447
AveOccup      0.138787
AveRooms      0.116850
Population    0.101218
AveBedrms     0.099248
HouseAge      0.071684
dtype: float64
R² на train с категориальным столбцом: 0.8412228991410066

Breast Cancer — AUC (My GBMClassifier): 0.9887640449438202
Breast Cancer — AUC (sklearn GBC):       0.9950062421972534
Accuracy My / sklearn: 0.958041958041958 / 0.958041958041958
